# Studi Kasus: Simulasi Berbasis Agen untuk Kebijakan Penghematan Listrik Gedung Kampus

Notebook ini menganalisis hasil sweep kebijakan yang dihasilkan oleh `src/sweep_parallel.py`
atau oleh SLURM array job di cluster HPC (`hpc/job.slurm`).

Keluaran yang dihitung:
1. Profil beban harian gedung (status quo vs kebijakan rekomendasi).
2. Frontier energi vs ketidaknyamanan untuk seluruh kebijakan.
3. Peta panas energi untuk setiap kombinasi jam mulai AC dan jam tutup lab.
4. Speedup paralel, efisiensi, dan kurva Hukum Amdahl dari `results/speedup.csv`.
5. Rekomendasi kebijakan beserta alasannya.

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / 'results').exists():
    ROOT = ROOT.parent
RESULTS = ROOT / 'results'
FIGURES = RESULTS / 'figures'
FIGURES.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(ROOT / 'src'))

import model  # noqa: E402

pd.set_option('display.float_format', '{:,.2f}'.format)
print('repo:', ROOT)

## 1. Muat hasil sweep

`sweep_results.csv` berisi satu baris per (kebijakan, pengulangan acak).

In [ ]:
raw = pd.read_csv(RESULTS / 'sweep_results.csv')
print(raw.shape)
raw.head()

In [ ]:
summary = pd.read_csv(RESULTS / 'policy_summary.csv')
baseline = summary[summary['is_baseline'] == 1]
display(baseline[['policy_id', 'mean_energy_kwh', 'mean_peak_kw',
                  'mean_discomfort_person_hours', 'mean_policy_violations',
                  'mean_critical_exposure_hours', 'mean_cost_idr']])
summary.head(10)

## 2. Profil beban harian (status quo vs rekomendasi)

In [ ]:
bld = model.load_building(ROOT / 'data')
policies = {p.policy_id: p for p in model.load_policies(ROOT / 'data')}
recommendation = json.loads((RESULTS / 'recommended_policy.json').read_text())
base_policy = next(p for p in policies.values() if p.is_baseline)
best_policy = policies[recommendation['policy_id']]

fig, ax = plt.subplots(figsize=(10, 4.5))
substeps = bld.params.substeps
for pol, color, label in ((base_policy, '#2b2d42', 'Status quo'),
                          (best_policy, '#06d6a0', 'Rekomendasi')):
    result = model.simulate(pol, bld, seed=20260101, rep=0, collect_profile=True)
    profile = np.array(result.profile_kw)
    hourly = profile.reshape(7, 24, substeps).mean(axis=2)
    daily = hourly.mean(axis=0)
    ax.plot(range(24), daily, marker='o', color=color, label=label)
ax.set_xlabel('Jam'); ax.set_ylabel('Daya gedung (kW)')
ax.set_title('Profil beban harian gedung (rata-rata 7 hari simulasi)')
ax.grid(alpha=0.25); ax.legend(frameon=False)
fig.tight_layout()

## 3. Frontier energi vs ketidaknyamanan

Setiap titik = satu kebijakan. Hijau = memenuhi batasan (tanpa paparan ruang
kritis dan ketidaknyamanan di bawah ambang), abu-abu = melanggar batasan.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
ok = summary[summary['feasible'] == 1]
bad = summary[summary['feasible'] == 0]
ax.scatter(bad['mean_discomfort_person_hours'], bad['mean_energy_kwh'],
           s=28, color='#8d99ae', alpha=0.55, label='Melanggar batasan')
ax.scatter(ok['mean_discomfort_person_hours'], ok['mean_energy_kwh'],
           s=34, color='#06d6a0', alpha=0.85, label='Memenuhi batasan')
if not baseline.empty:
    b = baseline.iloc[0]
    ax.scatter([b['mean_discomfort_person_hours']], [b['mean_energy_kwh']],
               s=170, marker='*', color='#2b2d42', label='Status quo')
best = summary.sort_values(['feasible', 'mean_energy_kwh'], ascending=[False, True]).iloc[0]
ax.scatter([best['mean_discomfort_person_hours']], [best['mean_energy_kwh']],
           s=190, marker='*', color='#ef233c', label='Rekomendasi')
ax.set_xlabel('Ketidaknyamanan (orang-jam/minggu)')
ax.set_ylabel('Energi (kWh/minggu)')
ax.set_title('Frontier kebijakan')
ax.grid(alpha=0.25); ax.legend(frameon=False)
fig.tight_layout()

## 4. Peta panas: jam mulai AC x jam tutup lab

Setiap sel menunjukkan rata-rata energi (kWh/minggu) untuk kombinasi
jam mulai AC dan jam tutup lab, pada setpoint terbaik.

In [ ]:
best_setpoint = summary.sort_values('mean_energy_kwh')['min_setpoint_c'].iloc[0]
sub = summary[summary['min_setpoint_c'] == best_setpoint]
pivot = sub.pivot_table(index='lab_close_hour', columns='ac_start_hour',
                       values='mean_energy_kwh', aggfunc='mean')
fig, ax = plt.subplots(figsize=(8, 5))
im = ax.imshow(pivot.values, cmap='YlGnBu_r', aspect='auto', origin='lower')
ax.set_xticks(range(len(pivot.columns)), [str(c) for c in pivot.columns])
ax.set_yticks(range(len(pivot.index)), [str(i) for i in pivot.index])
ax.set_xlabel('Jam mulai AC'); ax.set_ylabel('Jam tutup lab')
ax.set_title(f'Rata-rata energi (kWh/minggu) pada setpoint {best_setpoint:.0f} C')
for i in range(pivot.shape[0]):
    for j in range(pivot.shape[1]):
        ax.text(j, i, f'{pivot.values[i, j]:.0f}', ha='center', va='center', fontsize=9)
fig.colorbar(im, ax=ax, label='kWh/minggu')
fig.tight_layout()

## 5. Dampak tiap knob kebijakan

Rata-rata energi dan ketidaknyamanan dikelompokkan per nilai parameter.

In [ ]:
grid = summary[summary['is_baseline'] == 0]
for knob, title in (('ac_start_hour', 'Jam mulai AC'),
                    ('min_setpoint_c', 'Batas setpoint minimum (C)'),
                    ('lab_close_hour', 'Jam tutup lab')):
    grouped = grid.groupby(knob).agg(
        energi=('mean_energy_kwh', 'mean'),
        tidak_nyaman=('mean_discomfort_person_hours', 'mean'),
        manual=('mean_policy_violations', 'mean'),
        kritis=('mean_critical_exposure_hours', 'mean')).reset_index()
    fig, ax = plt.subplots(figsize=(7, 3.5))
    ax2 = ax.twinx()
    ax.bar(grouped[knob].astype(str), grouped['energi'], color='#3a86ff', label='energi')
    ax2.plot(grouped[knob].astype(str), grouped['tidak_nyaman'], 'o-', color='#ef233c',
             label='tidak nyaman')
    ax.set_title(f'{title}: energi dan ketidaknyamanan')
    ax.set_xlabel(title); ax.set_ylabel('kWh/minggu'); ax2.set_ylabel('orang-jam/minggu')
    fig.tight_layout()

## 6. Skala paralel dan Hukum Amdahl

Speedup = waktu serial / waktu paralel. Efisiensi = speedup / jumlah pekerja.
Kurva Amdahl: `S(N) = 1 / ((1 - f) + f/N)` dengan `f` = fraksi pekerjaan yang
bisa diparalelkan (di sini semua pekerjaan bisa, kecuali start up pool dan
komunikasi antar proses).

In [ ]:
speedup = pd.read_csv(RESULTS / 'speedup.csv')
ceiling_path = RESULTS / 'hardware_ceiling.csv'
ceiling = pd.read_csv(ceiling_path) if ceiling_path.exists() else None

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
ax = axes[0]
ax.plot(speedup['workers'], speedup['speedup'], 'o-', color='#ef233c', label='Speedup terukur')
ax.plot(speedup['workers'], speedup['steady_state_speedup'], 's--', color='#06d6a0',
        label='Tanpa biaya start up pool')
top = int(speedup['workers'].max())
ax.plot([1, top], [1, top], ':', color='#8d99ae', label='Linear ideal')
ax.plot(speedup['workers'], speedup['amdahl_predicted_speedup'], '--', color='#3a86ff',
        label=f"Amdahl (f={speedup['amdahl_f'].iloc[0]:.3f})")
if ceiling is not None:
    ax.plot(ceiling['workers'], ceiling['speedup'], '^:', color='#ffb703',
            label='Plafon mesin (CPU murni)')
ax.set_xlabel('Jumlah pekerja'); ax.set_ylabel('Speedup')
ax.set_title('Speedup paralel'); ax.grid(alpha=0.25); ax.legend(frameon=False, fontsize=9)

ax = axes[1]
ax.plot(speedup['workers'], speedup['efficiency'] * 100, 'o-', color='#ef233c',
        label='Efisiensi (speedup/N)')
if ceiling is not None:
    ax.plot(ceiling['workers'], ceiling['efficiency'] * 100, '^:', color='#ffb703',
            label='Efisiensi plafon mesin')
ax.set_xlabel('Jumlah pekerja'); ax.set_ylabel('Efisiensi (%)')
ax.set_title('Efisiensi paralel'); ax.grid(alpha=0.25); ax.legend(frameon=False)
fig.tight_layout()

In [ ]:
timings = pd.read_csv(RESULTS / 'timings.csv')
timings

### Titik impas: kapan paralel mulai menang?

Untuk pekerjaan kecil, biaya start up proses lebih besar daripada waktu
komputasinya, sehingga serial menang. Ini bentuk paling sering dari Hukum
Amdahl di dunia nyata.

In [ ]:
crossover_path = RESULTS / 'crossover.csv'
if crossover_path.exists():
    cross = pd.read_csv(crossover_path)
    display(cross)
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(cross['n_simulations'], cross['serial_time_s'], 'o-', color='#2b2d42', label='Serial')
    ax.plot(cross['n_simulations'], cross['parallel_time_s'], 's-', color='#06d6a0', label='Paralel (8 pekerja)')
    ax.set_xscale('log'); ax.set_xlabel('Jumlah simulasi'); ax.set_ylabel('Waktu (detik)')
    ax.set_title('Titik impas serial vs paralel'); ax.grid(alpha=0.25); ax.legend(frameon=False)
    fig.tight_layout()
else:
    print('crossover.csv belum ada; jalankan: python src/benchmark.py')

## 7. Rekomendasi kebijakan

Kandidat dipilih dari kebijakan yang memenuhi dua batasan nyata:
1. **Batasan keras** - ruang kritis (Lab Biologi) tidak boleh kehilangan pendinginan
   saat dipakai (`mean_critical_exposure_hours == 0`).
2. **Batasan lunak** - ketidaknyamanan total di bawah ambang
   (`mean_discomfort_person_hours`).

Di antara yang memenuhi batasan, dipilih yang energinya paling rendah.

In [ ]:
display(json.dumps(recommendation, indent=2))

In [ ]:
feasible = summary[summary['feasible'] == 1].sort_values('mean_energy_kwh')
feasible.head(10)[['policy_id', 'ac_start_hour', 'min_setpoint_c', 'lab_close_hour',
                   'mean_energy_kwh', 'saving_vs_baseline_pct',
                   'mean_discomfort_person_hours', 'mean_policy_violations',
                   'mean_critical_exposure_hours', 'mean_cost_idr']]

## 8. Kesimpulan

1. **Kebijakan hemat energi ada, tapi tidak gratis.** Menaikkan setpoint dan
   menunda penyalaan AC menghemat energi dengan sedikit ketidaknyamanan;
   batasan yang mengikat adalah ruang kritis, bukan kenyamanan.
2. **Batasan keras menentukan bentuk kebijakan.** Jam mulai AC yang terlalu
   lambat bertabrakan dengan operasional Lab Biologi yang mulai jam 7.
3. **Perilaku manusia mengubah hitungan.** AC yang dinyalakan manual oleh penghuni
   membuat kebijakan yang terlalu ketat tidak terlihat dampaknya pada tagihan.
4. **Paralelisasi hampir linear** karena setiap simulasi saling bebas. Tapi di
   laptop dengan CPU terbatas, perbaikan hanya sampai batas mesin - dan di
   situlah HPC masuk.